In [ ]:
!pip install -q groq fastapi uvicorn nest-asyncio

In [ ]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [ ]:
from groq import Groq
import os

client = Groq(
    api_key=os.environ["GROQ_API_KEY"],
    timeout=30.0,
    max_retries=1
)

MODEL = "openai/gpt-oss-20b"

print("Groq client initialized successfully!")
print("Selected model:", MODEL)

Groq client initialized successfully!
Selected model: openai/gpt-oss-20b


In [ ]:
SYSTEM_PROMPT = """
You are Byte, a friendly and patient programming mentor.

Your goal is to help beginners learn programming.

PERSONALITY:
- Be friendly, supportive, and encouraging.
- Be patient with beginners.
- Never make the user feel bad for asking basic questions.

TEACHING STYLE:
- Explain concepts in simple English.
- Assume the user has no computer science background.
- Break difficult concepts into small steps.
- Use real-world examples and simple analogies.
- Provide Python code examples whenever relevant.
- Explain code line by line when necessary.
- Avoid unnecessary technical jargon.

CONVERSATION:
- Remember information shared earlier in the conversation.
- Use the user's name when appropriate.
- Connect new explanations with previously discussed topics.
- Answer follow-up questions using conversation history.

IMPORTANT:
- Give accurate and practical explanations.
- If you are unsure, say so.
- Do not invent information.
"""

print("Byte's personality has been configured!")

Byte's personality has been configured!


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(
    title="Byte - Programming Mentor Chatbot",
    description="A multi-turn AI chatbot that helps beginners learn programming.",
    version="1.0.0"
)

class ChatRequest(BaseModel):
    message: str
    conversation_id: Optional[str] = "default"

class ChatResponse(BaseModel):
    response: str
    conversation_id: str

print("FastAPI application created successfully!")

FastAPI application created successfully!


In [ ]:
# Store conversation histories
conversation_histories = {}

def get_conversation(conversation_id):
    if conversation_id not in conversation_histories:
        conversation_histories[conversation_id] = []

    return conversation_histories[conversation_id]

print("Conversation history manager created successfully!")

Conversation history manager created successfully!


In [ ]:
from fastapi import HTTPException

@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):

    try:
        # Validate the message
        user_message = request.message.strip()

        if not user_message:
            raise HTTPException(
                status_code=400,
                detail="Message cannot be empty."
            )

        # Get conversation history
        history = get_conversation(request.conversation_id)

        # Add the system prompt and conversation history
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]

        messages.extend(history)

        # Add the latest user message
        messages.append({
            "role": "user",
            "content": user_message
        })

        # Send request to Groq
        completion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
            max_completion_tokens=2048
        )

        bot_response = completion.choices[0].message.content

        if not bot_response:
            raise HTTPException(
                status_code=502,
                detail="The AI returned an empty response."
            )

        # Save conversation history
        history.append({
            "role": "user",
            "content": user_message
        })

        history.append({
            "role": "assistant",
            "content": bot_response
        })

        return ChatResponse(
            response=bot_response,
            conversation_id=request.conversation_id
        )

    except HTTPException:
        raise

    except Exception as e:
        print("Error:", str(e))
        raise HTTPException(
            status_code=500,
            detail="Something went wrong while generating the response."
        )

print("Byte's /chat endpoint created successfully!")

Byte's /chat endpoint created successfully!


In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

print("Byte FastAPI server is running!")

Byte FastAPI server is running!


In [ ]:
from IPython.display import display, HTML

display(HTML("""
<h3>Byte API Documentation</h3>
<a href="http://127.0.0.1:8000/docs" target="_blank">
    Open FastAPI Swagger UI
</a>
"""))

In [ ]:
from google.colab import output
from IPython.display import display, HTML
import requests

# Check whether FastAPI is running
response = requests.get("http://127.0.0.1:8000/docs")

print("FastAPI status:", response.status_code)

# Create a Colab-accessible URL
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8000)")

# Open Swagger UI through Colab
docs_url = proxy_url.rstrip("/") + "/docs"

display(HTML(
    f'<a href="{docs_url}" target="_blank">'
    '<h3>Open Byte API Documentation</h3></a>'
))

INFO:     127.0.0.1:34388 - "GET /docs HTTP/1.1" 200 OK
FastAPI status: 200


In [ ]:
import requests
import time

time.sleep(2)

print("Thread running:", server_thread.is_alive())
print("Server started:", server.started)

response = requests.get("http://127.0.0.1:8000/docs")

print("Local API status:", response.status_code)

Thread running: True
Server started: True
INFO:     127.0.0.1:51826 - "GET /docs HTTP/1.1" 200 OK
Local API status: 200


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
import requests
import time

time.sleep(3)

try:
    response = requests.get("http://127.0.0.1:8000/docs", timeout=10)
    print("HTTP status:", response.status_code)
    print("FastAPI is responding:", response.status_code == 200)

except Exception as e:
    print("Connection failed:", repr(e))
    print("Server thread alive:", server_thread.is_alive())

INFO:     127.0.0.1:56098 - "GET /docs HTTP/1.1" 200 OK
HTTP status: 200
FastAPI is responding: True


In [ ]:
import requests

payload = {
    "message": "Hello Byte! Explain Python variables simply.",
    "conversation_id": "test_001"
}

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json=payload,
    timeout=60
)

print("Status code:", response.status_code)
print("Byte's response:", response.json())

INFO:     127.0.0.1:47846 - "POST /chat HTTP/1.1" 200 OK
Status code: 200
Byte's response: {'response': 'Hi there! 👋  \nLet’s talk about **Python variables**—the building blocks for storing and working with data.\n\n---\n\n## 1. What is a variable?\n\nThink of a variable like a **label on a box**.  \n- The **label** (the variable name) tells you what’s inside the box.  \n- The **box** (the variable itself) holds a value (a number, a word, a list, etc.).\n\nIn Python, you *create* a variable by giving it a name and assigning it a value.\n\n```python\n# Create a variable named age and give it the number 25\nage = 25\n```\n\nNow you can use `age` anywhere in your program, and Python will know you mean the value `25`.\n\n---\n\n## 2. How to make a variable\n\n```python\n# 1️⃣ Give it a name (no spaces, start with a letter or underscore)\nname = "Alice"\n\n# 2️⃣ Use the equals sign (=) to *assign* a value\nscore = 42\nis_valid = True\n```\n\n### Naming rules (simple ones)\n\n| Rule | Exampl

In [ ]:
import requests

url = "http://127.0.0.1:8000/chat"

messages = [
    "My name is Sree.",
    "I am a beginner learning Python.",
    "What is my name and which language am I learning?"
]

for message in messages:
    response = requests.post(
        url,
        json={
            "message": message,
            "conversation_id": "memory_test_001"
        },
        timeout=60
    )

    print("\nYou:", message)
    print("Byte:", response.json()["response"])
    print("-" * 60)

INFO:     127.0.0.1:46610 - "POST /chat HTTP/1.1" 200 OK

You: My name is Sree.
Byte: Hi Sree! 👋 I'm Byte, your friendly coding buddy. How can I help you today? Whether you’re just starting out or tackling a specific problem, I’m here to walk you through it step by step. Let me know what you’d like to learn or what question you have!
------------------------------------------------------------
INFO:     127.0.0.1:60284 - "POST /chat HTTP/1.1" 200 OK

You: I am a beginner learning Python.
Byte: Nice to meet you, Sree! 🎉  
Python is a great first language because it reads a lot like everyday English. Let’s start with the very basics so you feel comfortable writing and running your first few lines of code.

---

## 1. What is Python?

- **High‑level language** – you don’t have to worry about memory or hardware details.
- **Interpreted** – you write code, then run it, and it executes immediately.
- **Readable** – the syntax is clean and simple, so beginners can focus on logic.

---

## 2. 

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.7) ...
Setting up cloudflared (2026.10.7) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
import subprocess
import time
import re

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(30):
    line = tunnel_process.stdout.readline()
    if line:
        print(line.strip())
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            PUBLIC_URL = match.group(0)
            break

print("Public URL:", PUBLIC_URL if "PUBLIC_URL" in globals() else "Not found yet")

2026-11-08T14:12:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-08T14:12:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-08T14:12:52Z INF +--------------------------------------------------------------------------------------------+
2026-11-08T14:12:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-08T14:12:52Z INF |  https://portion-tries-broad-laws.trycloudflare.com   

In [ ]:
# Context-window configuration
MAX_HISTORY_MESSAGES = 10

def get_conversation(conversation_id):

    if conversation_id not in conversation_histories:
        conversation_histories[conversation_id] = []

    history = conversation_histories[conversation_id]

    # Keep only the most recent messages
    if len(history) > MAX_HISTORY_MESSAGES:
        history[:] = history[-MAX_HISTORY_MESSAGES:]

    # Ensure history begins with a user message
    if history and history[0]["role"] == "assistant":
        history.pop(0)

    return history

print("Context-window management enabled!")
print("Maximum retained messages:", MAX_HISTORY_MESSAGES)

Context-window management enabled!
Maximum retained messages: 10


In [ ]:
# Create a test conversation with 14 messages
test_history = []

for i in range(7):
    test_history.append({
        "role": "user",
        "content": f"User message {i+1}"
    })
    test_history.append({
        "role": "assistant",
        "content": f"Assistant response {i+1}"
    })

conversation_histories["context_test"] = test_history

print("Before truncation:", len(test_history), "messages")

# Apply context-window management
history = get_conversation("context_test")

print("After truncation:", len(history), "messages")
print("\nFirst retained message:", history[0])
print("Last retained message:", history[-1])

assert len(history) == 10
assert history[0]["role"] == "user"

print("\nContext-window test passed!")

Before truncation: 14 messages
After truncation: 10 messages

First retained message: {'role': 'user', 'content': 'User message 3'}
Last retained message: {'role': 'assistant', 'content': 'Assistant response 7'}

Context-window test passed!


In [ ]:
import sqlite3
from datetime import datetime

DB_NAME = "byte_chatbot.db"

def initialize_database():

    conn = sqlite3.connect(DB_NAME)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_conversation
        ON messages(conversation_id, id)
    """)

    conn.commit()
    conn.close()

initialize_database()

print("SQLite database initialized successfully!")
print("Database file:", DB_NAME)

SQLite database initialized successfully!
Database file: byte_chatbot.db


In [ ]:
from fastapi import HTTPException
import sqlite3

def load_history(conversation_id):
    """Load the latest 10 messages from SQLite."""

    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id DESC
            LIMIT ?
        """, (conversation_id, MAX_HISTORY_MESSAGES)).fetchall()

    # Restore chronological order
    history = [
        {"role": role, "content": content}
        for role, content in reversed(rows)
    ]

    # Ensure history starts with a user message
    if history and history[0]["role"] == "assistant":
        history.pop(0)

    return history


def save_messages(conversation_id, user_message, bot_response):
    """Save both messages in one database transaction."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.executemany("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, [
            (conversation_id, "user", user_message),
            (conversation_id, "assistant", bot_response)
        ])


def chat_with_sqlite(request: ChatRequest):

    try:
        user_message = request.message.strip()
        conversation_id = request.conversation_id or "default"

        if not user_message:
            raise HTTPException(
                status_code=400,
                detail="Message cannot be empty."
            )

        if len(user_message) > 4000:
            raise HTTPException(
                status_code=400,
                detail="Message is too long. Maximum 4000 characters."
            )

        # Retrieve conversation from SQLite
        history = load_history(conversation_id)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]

        messages.extend(history)

        messages.append({
            "role": "user",
            "content": user_message
        })

        # Call Groq
        completion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
            max_completion_tokens=2048
        )

        bot_response = completion.choices[0].message.content

        if not bot_response:
            raise HTTPException(
                status_code=502,
                detail="AI returned an empty response."
            )

        # Save conversation to SQLite
        save_messages(
            conversation_id,
            user_message,
            bot_response
        )

        return ChatResponse(
            response=bot_response,
            conversation_id=conversation_id
        )

    except HTTPException:
        raise

    except Exception as e:
        print("Chat error:", str(e))

        raise HTTPException(
            status_code=500,
            detail="Something went wrong while generating the response."
        )


# Replace the earlier POST /chat route
app.router.routes = [
    route for route in app.router.routes
    if not (
        getattr(route, "path", None) == "/chat"
        and "POST" in getattr(route, "methods", set())
    )
]

app.add_api_route(
    "/chat",
    chat_with_sqlite,
    methods=["POST"],
    response_model=ChatResponse
)

print("Byte is now connected to SQLite!")
print("POST /chat updated successfully!")

Byte is now connected to SQLite!
POST /chat updated successfully!


In [ ]:
import requests
import sqlite3

# Send a test message through FastAPI
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "My favourite programming language is Python.",
        "conversation_id": "sqlite_test_001"
    },
    timeout=60
)

print("API status:", response.status_code)
print("Byte:", response.json()["response"])

# Inspect the database
conn = sqlite3.connect(DB_NAME)

rows = conn.execute("""
    SELECT conversation_id, role, content, created_at
    FROM messages
    WHERE conversation_id = ?
    ORDER BY id
""", ("sqlite_test_001",)).fetchall()

conn.close()

print("\nMessages stored in SQLite:")

for row in rows:
    print(f"{row[1]}: {row[2]}")

print("\nTotal messages saved:", len(rows))

INFO:     127.0.0.1:58458 - "POST /chat HTTP/1.1" 200 OK
API status: 200
Byte: That’s awesome! Python is a great choice—so many people love its simplicity and power. 🌟

What do you enjoy most about Python? Are you working on a particular project, or maybe you’d like to learn something new (like data analysis, web development, or just more Python tricks)? Let me know what you’re curious about, and I’ll help you dive in!

Messages stored in SQLite:

Total messages saved: 0


In [ ]:
import os
import sqlite3

print("Database location:", os.path.abspath(DB_NAME))
print("Database exists:", os.path.exists(DB_NAME))

print("\nRegistered POST /chat routes:")

for route in app.routes:
    if getattr(route, "path", None) == "/chat":
        print("Endpoint:", route.endpoint.__name__)

print("\nMessages in database:")

with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT conversation_id, role, content
        FROM messages
        ORDER BY id DESC
        LIMIT 10
    """).fetchall()

    print("Total messages:", len(rows))

    for row in rows:
        print(row)

Database location: /content/byte_chatbot.db
Database exists: True

Registered POST /chat routes:
Endpoint: chat_with_sqlite

Messages in database:
Total messages: 0


In [ ]:
# Test database writing without calling the API

save_messages(
    "manual_test_001",
    "Hello from SQLite test",
    "Hello! Database connection works."
)

with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, ("manual_test_001",)).fetchall()

print("Saved messages:", len(rows))

for row in rows:
    print(row)

assert len(rows) == 2, "SQLite save test failed!"

print("\nSQLite save function works!")

Saved messages: 2
('user', 'Hello from SQLite test')
('assistant', 'Hello! Database connection works.')

SQLite save function works!


In [ ]:
# Add a diagnostic version of the SQLite endpoint

def chat_with_sqlite(request: ChatRequest):

    try:
        user_message = request.message.strip()
        conversation_id = request.conversation_id or "default"

        if not user_message:
            raise HTTPException(status_code=400, detail="Message cannot be empty.")

        print("SQLite endpoint called:", conversation_id)

        history = load_history(conversation_id)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + history + [
            {"role": "user", "content": user_message}
        ]

        completion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
            max_completion_tokens=2048
        )

        bot_response = completion.choices[0].message.content

        if not bot_response:
            raise HTTPException(status_code=502, detail="Empty AI response.")

        print("Saving messages to SQLite...")

        save_messages(conversation_id, user_message, bot_response)

        print("SQLite save completed!")

        return ChatResponse(
            response=bot_response,
            conversation_id=conversation_id
        )

    except HTTPException:
        raise

    except Exception as e:
        print("Chat error:", repr(e))
        raise HTTPException(status_code=500, detail="Chat processing failed.")


# Update the existing route
for route in app.routes:
    if getattr(route, "path", None) == "/chat" and "POST" in getattr(route, "methods", set()):
        route.endpoint = chat_with_sqlite

print("Diagnostic SQLite endpoint updated!")

Diagnostic SQLite endpoint updated!


In [ ]:
# Remove the existing POST /chat route
app.router.routes = [
    route for route in app.router.routes
    if not (
        getattr(route, "path", None) == "/chat"
        and "POST" in getattr(route, "methods", set())
    )
]

# Register the new SQLite endpoint
app.add_api_route(
    "/chat",
    chat_with_sqlite,
    methods=["POST"],
    response_model=ChatResponse
)

print("Updated SQLite endpoint registered correctly!")

Updated SQLite endpoint registered correctly!


In [ ]:
import requests
import sqlite3

test_id = "sqlite_api_test_002"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "Remember that I am testing SQLite persistence.",
        "conversation_id": test_id
    },
    timeout=60
)

print("HTTP status:", response.status_code)
print("Byte:", response.json().get("response"))

with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, (test_id,)).fetchall()

print("\nMessages saved:", len(rows))

for role, content in rows:
    print(role, ":", content[:150])

INFO:     127.0.0.1:60700 - "POST /chat HTTP/1.1" 200 OK
HTTP status: 200
Byte: Got it! I’ll keep in mind that you’re testing SQLite persistence as we chat. If you need any help or have questions about SQLite, feel free to let me know!

Messages saved: 0


In [ ]:
import requests
import sqlite3

test_id = "sqlite_api_test_002"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "Remember that I am testing SQLite persistence.",
        "conversation_id": test_id
    },
    timeout=60
)

print("HTTP status:", response.status_code)
print("Byte:", response.json().get("response"))

with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, (test_id,)).fetchall()

print("\nMessages saved:", len(rows))

for role, content in rows:
    print(role, ":", content[:150])

INFO:     127.0.0.1:37636 - "POST /chat HTTP/1.1" 200 OK
HTTP status: 200
Byte: Absolutely! I’ll keep that in mind. Whenever you need help or have a question about SQLite persistence, just let me know. Happy coding!

Messages saved: 0


In [ ]:
import time
import threading
import uvicorn
from fastapi import FastAPI

# Stop the old FastAPI server
server.should_exit = True
server_thread.join(timeout=10)

print("Old server stopped:", not server_thread.is_alive())

# Create a fresh FastAPI application
app = FastAPI(
    title="Byte - Programming Mentor Chatbot",
    description="Multi-turn AI programming mentor",
    version="1.0.0"
)

# Register the SQLite endpoint on the fresh application
app.add_api_route(
    "/chat",
    chat_with_sqlite,
    methods=["POST"],
    response_model=ChatResponse
)

# Start a fresh server
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("New server started:", server.started)
print("SQLite endpoint:", chat_with_sqlite.__name__)

Old server stopped: True


INFO:     Shutting down
INFO:     Started server process [721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [721]


New server started: True
SQLite endpoint: chat_with_sqlite


Test persistence again

In [ ]:
import requests
import sqlite3

test_id = "sqlite_final_test_001"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "My name is Sree. Please remember this for my SQLite test.",
        "conversation_id": test_id
    },
    timeout=60
)

print("HTTP status:", response.status_code)

with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, (test_id,)).fetchall()

print("Messages saved:", len(rows))

for role, content in rows:
    print(f"{role}: {content[:150]}")

assert response.status_code == 200
assert len(rows) == 2

print("\nSQLite API persistence test PASSED!")

SQLite endpoint called: sqlite_final_test_001
Saving messages to SQLite...
SQLite save completed!
INFO:     127.0.0.1:43434 - "POST /chat HTTP/1.1" 200 OK
HTTP status: 200
Messages saved: 2
user: My name is Sree. Please remember this for my SQLite test.
assistant: Nice to meet you, Sree! I’ve got your name down—just let me know what you’d like to work on for your SQLite test, and we’ll tackle it together. Whethe

SQLite API persistence test PASSED!


In [ ]:
import requests
import sqlite3

test_id = "sqlite_final_test_001"

# Ask a follow-up question using the same conversation ID
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "What is my name?",
        "conversation_id": test_id
    },
    timeout=60
)

print("HTTP status:", response.status_code)
print("\nByte:", response.json()["response"])

# Check messages stored in SQLite
with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, (test_id,)).fetchall()

print("\nTotal messages in SQLite:", len(rows))

SQLite endpoint called: sqlite_final_test_001
Saving messages to SQLite...
SQLite save completed!
INFO:     127.0.0.1:56816 - "POST /chat HTTP/1.1" 200 OK
HTTP status: 200

Byte: Your name is **Sree**!

Total messages in SQLite: 4


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

# Create a folder for Byte's database
DB_FOLDER = "/content/drive/MyDrive/Byte_Chatbot"

os.makedirs(DB_FOLDER, exist_ok=True)

# Permanent database location
NEW_DB_PATH = os.path.join(DB_FOLDER, "byte_chatbot.db")

# Move the existing database without overwriting an existing Drive database
if os.path.exists(DB_NAME):

    if not os.path.exists(NEW_DB_PATH):
        shutil.copy2(DB_NAME, NEW_DB_PATH)
        print("Existing database copied to Google Drive!")

    else:
        print("A database already exists in Google Drive.")
        print("We will preserve it rather than overwrite it.")

else:
    print("Local database not found. Check the current runtime.")

DB_NAME = NEW_DB_PATH

print("Database location:", DB_NAME)
print("Database exists:", os.path.exists(DB_NAME))

Existing database copied to Google Drive!
Database location: /content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db
Database exists: True


In [ ]:
import sqlite3
import os

with sqlite3.connect(DB_NAME) as conn:

    total = conn.execute(
        "SELECT COUNT(*) FROM messages"
    ).fetchone()[0]

    conversations = conn.execute("""
        SELECT DISTINCT conversation_id
        FROM messages
    """).fetchall()

print("Database exists:", os.path.exists(DB_NAME))
print("Total stored messages:", total)
print("Stored conversations:")

for conversation in conversations:
    print("-", conversation[0])

Database exists: True
Total stored messages: 6
Stored conversations:
- manual_test_001
- sqlite_final_test_001


In [ ]:
import os
import sqlite3

# Confirm Drive database path
DB_NAME = "/content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db"

# Verify the database is accessible
assert os.path.exists(DB_NAME), "Drive database not found!"

# Verify database connection
with sqlite3.connect(DB_NAME) as conn:
    result = conn.execute(
        "SELECT COUNT(*) FROM messages"
    ).fetchone()[0]

print("Byte database connected to Google Drive!")
print("Database:", DB_NAME)
print("Existing messages:", result)

Byte database connected to Google Drive!
Database: /content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db
Existing messages: 6


In [ ]:
import requests
import sqlite3

test_id = "google_drive_test_001"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "Remember that I am testing Google Drive database storage.",
        "conversation_id": test_id
    },
    timeout=60
)

print("API status:", response.status_code)
print("Byte:", response.json().get("response"))

# Check the Google Drive database
with sqlite3.connect(DB_NAME) as conn:
    rows = conn.execute("""
        SELECT role, content
        FROM messages
        WHERE conversation_id = ?
        ORDER BY id
    """, (test_id,)).fetchall()

print("\nMessages saved in Google Drive:", len(rows))

for role, content in rows:
    print(role, ":", content[:150])

assert response.status_code == 200
assert len(rows) == 2

print("\nGoogle Drive persistence test PASSED!")

SQLite endpoint called: google_drive_test_001
Saving messages to SQLite...
SQLite save completed!
INFO:     127.0.0.1:36430 - "POST /chat HTTP/1.1" 200 OK
API status: 200
Byte: Got it! I’ll keep in mind that you’re testing Google Drive database storage. If you have any questions about how to work with Google Drive, how to store data, or anything else, just let me know—I’m here to help!

Messages saved in Google Drive: 2
user : Remember that I am testing Google Drive database storage.
assistant : Got it! I’ll keep in mind that you’re testing Google Drive database storage. If you have any questions about how to work with Google Drive, how to sto

Google Drive persistence test PASSED!


In [ ]:
import requests
import sqlite3

test_id = "google_drive_test_001"

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "What was I testing?",
        "conversation_id": test_id
    },
    timeout=60
)

print("API status:", response.status_code)
print("Byte:", response.json()["response"])

with sqlite3.connect(DB_NAME) as conn:
    count = conn.execute("""
        SELECT COUNT(*)
        FROM messages
        WHERE conversation_id = ?
    """, (test_id,)).fetchone()[0]

print("\nMessages stored in Drive:", count)

SQLite endpoint called: google_drive_test_001
Saving messages to SQLite...
SQLite save completed!
INFO:     127.0.0.1:43112 - "POST /chat HTTP/1.1" 200 OK
API status: 200
Byte: You were testing Google Drive database storage.

Messages stored in Drive: 4


In [ ]:
import os
import sqlite3

print("Database exists:", os.path.exists(DB_NAME))
print("Database path:", DB_NAME)

with sqlite3.connect(DB_NAME) as conn:
    count = conn.execute(
        "SELECT COUNT(*) FROM messages"
    ).fetchone()[0]

print("Total messages in Drive:", count)

Database exists: True
Database path: /content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db
Total messages in Drive: 10


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sqlite3

DB_NAME = "/content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db"

assert os.path.exists(DB_NAME), "Database not found!"

with sqlite3.connect(DB_NAME) as conn:
    total = conn.execute(
        "SELECT COUNT(*) FROM messages"
    ).fetchone()[0]

    conversations = conn.execute("""
        SELECT DISTINCT conversation_id
        FROM messages
    """).fetchall()

print("Database recovered successfully!")
print("Total messages:", total)
print("Conversations:", [c[0] for c in conversations])

Database recovered successfully!
Total messages: 10
Conversations: ['google_drive_test_001', 'manual_test_001', 'sqlite_final_test_001']


In [ ]:
!pip install -q groq fastapi uvicorn nest-asyncio

In [ ]:
from google.colab import userdata
from groq import Groq
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(
    api_key=GROQ_API_KEY,
    timeout=30.0,
    max_retries=1
)

MODEL = "openai/gpt-oss-20b"

print("Groq reconnected successfully!")

Groq reconnected successfully!


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import sqlite3

DB_NAME = "/content/drive/MyDrive/Byte_Chatbot/byte_chatbot.db"

MAX_HISTORY_MESSAGES = 10

SYSTEM_PROMPT = """
You are Byte, a friendly and patient programming mentor.

Your goal is to help beginners learn programming.

- Explain concepts in simple English.
- Assume the user has no computer science background.
- Break difficult concepts into small steps.
- Use real-world examples and Python code.
- Remember earlier messages in the same conversation.
- Be accurate and never invent information.
"""

class ChatRequest(BaseModel):
    message: str
    conversation_id: Optional[str] = "default"

class ChatResponse(BaseModel):
    response: str
    conversation_id: str

app = FastAPI(
    title="Byte - Programming Mentor Chatbot",
    description="A multi-turn AI programming mentor",
    version="1.0.0"
)

print("Byte configuration restored!")

Byte configuration restored!


In [ ]:
def load_history(conversation_id):

    with sqlite3.connect(DB_NAME) as conn:
        rows = conn.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id DESC
            LIMIT ?
        """, (conversation_id, MAX_HISTORY_MESSAGES)).fetchall()

    history = [
        {"role": role, "content": content}
        for role, content in reversed(rows)
    ]

    if history and history[0]["role"] == "assistant":
        history.pop(0)

    return history


def save_messages(conversation_id, user_message, bot_response):

    with sqlite3.connect(DB_NAME) as conn:
        conn.executemany("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, [
            (conversation_id, "user", user_message),
            (conversation_id, "assistant", bot_response)
        ])


print("SQLite functions restored!")

SQLite functions restored!


In [ ]:
def chat_with_sqlite(request: ChatRequest):

    try:
        user_message = request.message.strip()
        conversation_id = request.conversation_id or "default"

        if not user_message:
            raise HTTPException(400, "Message cannot be empty.")

        if len(user_message) > 4000:
            raise HTTPException(400, "Message is too long.")

        history = load_history(conversation_id)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + history + [
            {"role": "user", "content": user_message}
        ]

        completion = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.7,
            max_completion_tokens=2048
        )

        bot_response = completion.choices[0].message.content

        if not bot_response:
            raise HTTPException(502, "AI returned an empty response.")

        save_messages(
            conversation_id,
            user_message,
            bot_response
        )

        return ChatResponse(
            response=bot_response,
            conversation_id=conversation_id
        )

    except HTTPException:
        raise

    except Exception as e:
        print("Chat error:", repr(e))
        raise HTTPException(500, "Chat processing failed.")


app.add_api_route(
    "/chat",
    chat_with_sqlite,
    methods=["POST"],
    response_model=ChatResponse
)

print("Byte API restored successfully!")

Byte API restored successfully!


In [ ]:
import nest_asyncio
import threading
import time
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("Server running:", server.started)

INFO:     Started server process [721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server running: False


In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={
        "message": "What was I testing?",
        "conversation_id": "google_drive_test_001"
    },
    timeout=60
)

print("HTTP status:", response.status_code)
print("Byte:", response.json()["response"])

SQLite endpoint called: google_drive_test_001
Saving messages to SQLite...
SQLite save completed!
INFO:     127.0.0.1:35042 - "POST /chat HTTP/1.1" 200 OK
HTTP status: 200
Byte: You were testing Google Drive database storage.


In [54]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8000/docs",
        timeout=10
    )

    print("Existing server status:", response.status_code)

    if response.status_code == 200:
        print("FastAPI is already running!")
    else:
        print("Unexpected response:", response.status_code)

except Exception as e:
    print("Server check failed:", repr(e))

INFO:     127.0.0.1:58292 - "GET /docs HTTP/1.1" 200 OK
Existing server status: 200
FastAPI is already running!
